# Notebook 11 — `11_instance_mixed_effect_poverty_tpt`

## Overview

**Purpose:** Companion to the growth mixed-effects notebook, applying the same
random-intercept design to the two other RQ2 outcomes — poverty and unemployment
(TPT). The check confirms that the fixed-effects conclusions (poverty null;
unemployment suggestive/fragile) are stable across the fixed-versus-random
modelling choice. It does not turn either result into an effect — that stability
is the point. This is an inferential diagnostic (β, SE, ICC, AIC/BIC), not a
prediction model.

## Data requirements

This notebook runs on the model-ready panel exported by Notebook 8; no raw data
download is required here.

| Input | Produced by | Directory |
|---|---|---|
| Model-ready panel (`transformed` sheet) | Notebook 8 | `data/processed/8_panel_data.xlsx` |

## Workflow

| Step | Description |
|---|---|
| 1 | Assumptions checked before fitting (grouping, ICC rationale, VIF) |
| 2 | Reusable FE + mixed estimation function (flood-block joint Wald on both) |
| 3 | Results per outcome — poverty and unemployment |
| 4 | Model comparison: FE vs mixed (agreement on the flood conclusion) |
| 5 | Between- vs within-regency variation |
| 6 | Evaluation: post-fit diagnostics |
| 7 | What to report |

## Outputs

No files are written; the notebook reports the FE-vs-mixed comparison and the
flood-block joint tests for the manuscript's robustness paragraph.

In [3]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
from linearmodels.panel import PanelOLS

panel = pd.read_excel("../data/processed/8_panel_data.xlsx", sheet_name="transformed").sort_values(
    ["bps_kab_code", "year"]).reset_index(drop=True)

FLOOD    = ["event_count_log", "HCI_log", "PDI_log"]
CONTROLS = ["log_population", "rls_years"]
X        = FLOOD + CONTROLS
OUTCOMES = {"poverty_pct": "Poverty rate (%)",
            "tpt_pct":     "Unemployment rate / TPT (%)"}

print(f"Panel: {len(panel):,} rows, {panel.bps_kab_code.nunique()} regencies, "
      f"{panel.year.nunique()} years")
print("Outcomes:", ", ".join(OUTCOMES))

Panel: 5,140 rows, 514 regencies, 10 years
Outcomes: poverty_pct, tpt_pct


## STEP 1 — Assumptions checked before fitting

A random-intercept mixed model relies on a handful of conditions. These are
reported up front so the diagnostic is transparent rather than assumed. The
random effect itself is only justified if there is meaningful between-regency
variance (ICC > 0, computed per outcome below); VIF confirms the fixed-effect
predictors are not collinear (random effects do not fix multicollinearity).

In [5]:
# ═════════════════════════════════════════════════════════════
# STEP 1 — Assumptions checked before fitting
# ═════════════════════════════════════════════════════════════
print("MIXED-EFFECTS ASSUMPTIONS (random regency intercept, year fixed)")
print("=" * 66)

# A1. Balanced grouping
g = panel.groupby("bps_kab_code").size()
print(f"A1 grouping     : {panel.bps_kab_code.nunique()} regencies, "
      f"{g.min()}-{g.max()} obs each "
      f"({'balanced' if g.min()==g.max() else 'UNBALANCED'})")

# A2. Random effect justified only if ICC > 0 (reported per outcome in section 3)
print("A2 random effect: justified only if ICC > 0  (see per-outcome results)")

# A3. Fixed-effect predictors not collinear
xc  = sm.add_constant(panel[X].dropna())
vif = {c: variance_inflation_factor(xc.values, i)
       for i, c in enumerate(xc.columns) if c != "const"}
print("A3 VIF (<5 ok)  : " + ", ".join(f"{k}={v:.2f}" for k, v in vif.items()))

# A4/A5 checked post-fit / documented
print("A4 residual normality & homoskedasticity : checked post-fit (section 5)")
print("A5 MAR missingness, pre-imputed outcomes  : documented in Data section")

MIXED-EFFECTS ASSUMPTIONS (random regency intercept, year fixed)
A1 grouping     : 514 regencies, 10-10 obs each (balanced)
A2 random effect: justified only if ICC > 0  (see per-outcome results)
A3 VIF (<5 ok)  : event_count_log=3.18, HCI_log=1.21, PDI_log=3.31, log_population=1.19, rls_years=1.01
A4 residual normality & homoskedasticity : checked post-fit (section 5)
A5 MAR missingness, pre-imputed outcomes  : documented in Data section


## STEP 2 — Reusable estimation function

One function fits both FE and mixed for a given outcome, so poverty and TPT are
handled identically. Predictors are standardised **for optimiser stability only**;
reported coefficients are converted back to original units. The mixed model is fit
by ML (`reml=False`) so that AIC/BIC are well defined and comparable.

The **flood-block joint Wald test** (H0: all three flood coefficients = 0) is
computed on both the FE and the mixed fit — this is the test that speaks to the
flood hypothesis, as opposed to the per-coefficient control significances.

In [7]:
# ═════════════════════════════════════════════════════════════
# STEP 2 — Reusable estimation function
# ═════════════════════════════════════════════════════════════
def fit_fe_mixed(Y):
    """Fit two-way FE and random-intercept mixed model for outcome Y.
    Returns both fits plus ICC, back-scaling map, solver, and flood-block
    joint Wald results (FE and mixed)."""
    pdat = panel.set_index(["bps_kab_code", "year"])
    sub  = pdat[[Y] + X].dropna()

    # --- Fixed effects (two-way, clustered SE) — main inferential model ---
    fe = PanelOLS(sub[Y], sub[X], entity_effects=True, time_effects=True,
                  drop_absorbed=True).fit(cov_type="clustered", cluster_entity=True)

    # --- Mixed effects (random regency intercept, year fixed) ---
    df = sub.reset_index().copy()
    for c in X:
        df[c + "_z"] = (df[c] - df[c].mean()) / df[c].std()
    XZ = [c + "_z" for c in X]

    me = None
    for _m in ["lbfgs", "powell", "nm"]:
        try:
            _f = smf.mixedlm(f"{Y} ~ {' + '.join(XZ)} + C(year)",
                             data=df, groups=df["bps_kab_code"]).fit(
                             method=_m, reml=False)   # ML -> valid AIC/BIC
            if _f.converged:
                me, method = _f, _m
                break
        except Exception:
            pass
    assert me is not None, f"mixed model failed for {Y}"

    var_re = float(me.cov_re.iloc[0, 0]); var_e = float(me.scale)
    icc    = var_re / (var_re + var_e)
    scale  = {c: df[c].std() for c in X}

    # --- flood-block joint Wald: H0 all flood coefs = 0 (controls held fixed) ---
    w_fe   = fe.wald_test(formula=" = ".join(FLOOD) + " = 0")
    p_fe_b = float(w_fe.pval)

    fz   = [c + "_z" for c in FLOOD]              # H0 all=0 is scale-invariant
    b    = me.params[fz].values
    cov  = me.cov_params().loc[fz, fz].values
    wmix = float(b @ np.linalg.inv(cov) @ b)
    p_mix_b = float(stats.chi2.sf(wmix, len(fz)))

    return dict(Y=Y, fe=fe, me=me, icc=icc, scale=scale, method=method,
                chi2_fe=float(w_fe.stat), p_fe_blk=p_fe_b,
                chi2_mix=wmix, p_mix_blk=p_mix_b)

## STEP 3 — Results per outcome

For each outcome: FE vs mixed coefficient table (flood variables first, since
they are the research question), per-coefficient significance, an automatic
"very similar / moderately diff" stability flag, ICC, model fit (AIC/BIC), and
— most importantly — the **flood-block joint test** under both estimators.

Read the flood-block line, not the control stars: a `*` on `log_population` or
`rls_years` is a between-regency artefact, not a flood effect.

In [9]:
# ═════════════════════════════════════════════════════════════
# STEP 3 — Results per outcome
# ═════════════════════════════════════════════════════════════
def report(res):
    fe, me, Y = res["fe"], res["me"], res["Y"]
    label = OUTCOMES[Y]; scale = res["scale"]
    print("=" * 66)
    print(f"OUTCOME: {label}   [{Y}]   (mixed via {res['method']}, ML)")
    print("=" * 66)
    print(f"{'variable':<16}{'FE b':>9}{'FE p':>8}{'Mix b':>9}{'Mix p':>8}  status")
    print("-" * 60)
    for c in X:
        b_fe, p_fe = fe.params[c], fe.pvalues[c]
        b_me = me.params[c + "_z"] / scale[c]
        p_me = me.pvalues[c + "_z"]
        ad = abs(b_fe - b_me)
        status = ("very similar" if ad < 0.05
                  else "moderately diff" if ad < 0.20 else "subst. diff")
        tag  = "flood" if c in FLOOD else "control"
        s_fe = "*" if p_fe < 0.05 else " "
        s_me = "*" if p_me < 0.05 else " "
        print(f"{c:<16}{b_fe:>+9.3f}{p_fe:>7.3f}{s_fe}"
              f"{b_me:>+9.3f}{p_me:>7.3f}{s_me} {status} [{tag}]")

    print(f"\n  ICC = {res['icc']:.3f}    within-R2 (FE) = {fe.rsquared_within:.4f}")
    print(f"  AIC = {me.aic:.1f}   BIC = {me.bic:.1f}   logLik = {me.llf:.1f}")
    print(f"\n  [FLOOD-BLOCK joint   H0: {' = '.join(FLOOD)} = 0]")
    print(f"     FE    : chi2={res['chi2_fe']:7.3f}  p={res['p_fe_blk']:.4g}  "
          f"{'sig' if res['p_fe_blk']<0.05 else 'n.s.'}")
    print(f"     Mixed : chi2={res['chi2_mix']:7.3f}  p={res['p_mix_blk']:.4g}  "
          f"{'sig' if res['p_mix_blk']<0.05 else 'n.s.'}")
    agree = (res['p_fe_blk'] < 0.05) == (res['p_mix_blk'] < 0.05)
    print(f"     -> flood conclusion {'AGREES' if agree else 'DIFFERS'} "
          f"across FE vs mixed  <<< report this")
    print()
    return res

res_pov = report(fit_fe_mixed("poverty_pct"))
res_tpt = report(fit_fe_mixed("tpt_pct"))

OUTCOME: Poverty rate (%)   [poverty_pct]   (mixed via lbfgs, ML)
variable             FE b    FE p    Mix b   Mix p  status
------------------------------------------------------------
event_count_log    +0.002  0.352    +0.002  0.328  very similar [flood]
HCI_log            -0.001  0.532    -0.001  0.489  very similar [flood]
PDI_log            -0.000  0.373    -0.000  0.343  very similar [flood]
log_population     -0.041  0.178    -0.069  0.000* very similar [control]
rls_years          +0.000  0.991    -0.023  0.000* very similar [control]

  ICC = 0.988    within-R2 (FE) = 0.0313
  AIC = -11763.4   BIC = -11652.1   logLik = 5898.7

  [FLOOD-BLOCK joint   H0: event_count_log = HCI_log = PDI_log = 0]
     FE    : chi2=  1.853  p=0.6036  n.s.
     Mixed : chi2=  2.231  p=0.5259  n.s.
     -> flood conclusion AGREES across FE vs mixed  <<< report this

OUTCOME: Unemployment rate / TPT (%)   [tpt_pct]   (mixed via lbfgs, ML)
variable             FE b    FE p    Mix b   Mix p  status
--

OUTCOME: Unemployment rate / TPT (%)   [tpt_pct]   (mixed via lbfgs, ML)
variable             FE b    FE p    Mix b   Mix p  status
------------------------------------------------------------
event_count_log    -0.015  0.077    -0.012  0.128  very similar [flood]
HCI_log            +0.004  0.310    +0.004  0.266  very similar [flood]
PDI_log            +0.001  0.711    +0.001  0.358  very similar [flood]
log_population     -0.016  0.921    +0.133  0.000* moderately diff [control]
rls_years          +0.011  0.755    +0.118  0.000* moderately diff [control]

  ICC = 0.657    within-R2 (FE) = -0.0037
  AIC = 397.4   BIC = 508.6   logLik = -181.7

  [FLOOD-BLOCK joint   H0: event_count_log = HCI_log = PDI_log = 0]
     FE    : chi2=  5.958  p=0.1137  n.s.
     Mixed : chi2=  4.272  p=0.2336  n.s.
     -> flood conclusion AGREES across FE vs mixed  <<< report this



## STEP 4 — Model comparison: FE vs mixed

A compact side-by-side of the two estimators for each outcome. FE remains the
main inferential specification (justified by the Mundlak test in the FE
notebooks); the mixed model is the robustness comparator. What we want to see is
**agreement on the flood conclusion** — not identical control coefficients.

In [11]:
# ═════════════════════════════════════════════════════════════
# STEP 4 — Model comparison: FE vs mixed
# ═════════════════════════════════════════════════════════════
def compare(res):
    fe, me = res["fe"], res["me"]
    Y = res["Y"]
    print(f"--- {OUTCOMES[Y]} [{Y}] ---")
    print(f"  {'':22}{'FE (within)':>16}{'Mixed (RE intercept)':>22}")
    print(f"  {'estimator role':22}{'main inferential':>16}{'robustness check':>22}")
    print(f"  {'regency effect':22}{'fixed (absorbed)':>16}{'random intercept':>22}")
    print(f"  {'flood-block p':22}{res['p_fe_blk']:>16.4g}{res['p_mix_blk']:>22.4g}")
    print(f"  {'flood-block sig':22}{('sig' if res['p_fe_blk']<0.05 else 'n.s.'):>16}"
          f"{('sig' if res['p_mix_blk']<0.05 else 'n.s.'):>22}")
    print(f"  {'ICC':22}{'—':>16}{res['icc']:>22.3f}")
    agree = (res['p_fe_blk']<0.05)==(res['p_mix_blk']<0.05)
    print(f"  -> {'AGREE: FE conclusion stable' if agree else 'DIFFER: investigate'}")
    print()

compare(res_pov)
compare(res_tpt)

--- Poverty rate (%) [poverty_pct] ---
                             FE (within)  Mixed (RE intercept)
  estimator role        main inferential      robustness check
  regency effect        fixed (absorbed)      random intercept
  flood-block p                   0.6036                0.5259
  flood-block sig                   n.s.                  n.s.
  ICC                                  —                 0.988
  -> AGREE: FE conclusion stable

--- Unemployment rate / TPT (%) [tpt_pct] ---
                             FE (within)  Mixed (RE intercept)
  estimator role        main inferential      robustness check
  regency effect        fixed (absorbed)      random intercept
  flood-block p                  0.04374                0.2293
  flood-block sig                    sig                  n.s.
  ICC                                  —                 0.697
  -> DIFFER: investigate



## STEP 5 — Between- vs within-regency variation

Explains any movement in the **control** coefficients between FE and mixed —
identical logic to the growth notebook. Variables dominated by between-regency
variation (population, schooling) are partly absorbed by FE but not by the random
intercept, so their coefficients can differ across the two models. The flood
variables, by contrast, carry substantial within-regency variation, which is why
their coefficients stay close across estimators.

In [13]:
# ═════════════════════════════════════════════════════════════
# STEP 5 — Between- vs within-regency variation
# ═════════════════════════════════════════════════════════════
print("BETWEEN- vs WITHIN-regency variation (SD)")
print("=" * 60)
print(f"{'variable':<18}{'between':>9}{'within':>9}{'ratio':>8}  reading")
print("-" * 60)
_base = panel[["bps_kab_code"] + X].dropna()
for c in X:
    within_sd  = _base.groupby("bps_kab_code")[c].transform(lambda s: s - s.mean()).std()
    between_sd = _base.groupby("bps_kab_code")[c].mean().std()
    ratio = between_sd / max(within_sd, 1e-9)
    reading = ("mostly between" if ratio > 5
               else "substantial within" if ratio < 1.5 else "mixed")
    tag = "flood" if c in FLOOD else "control"
    print(f"{c:<18}{between_sd:>9.3f}{within_sd:>9.3f}{ratio:>7.1f}x  {reading} [{tag}]")

BETWEEN- vs WITHIN-regency variation (SD)
variable            between   within   ratio  reading
------------------------------------------------------------
event_count_log       0.570    0.580    1.0x  substantial within [flood]
HCI_log               0.386    0.924    0.4x  substantial within [flood]
PDI_log               3.091    3.637    0.8x  substantial within [flood]
log_population        1.022    0.073   14.0x  mostly between [control]
rls_years             1.630    0.347    4.7x  mixed [control]


PDI_log               3.091    3.637    0.8x  substantial within [flood]
log_population        1.022    0.073   14.0x  mostly between [control]
rls_years             1.630    0.347    4.7x  mixed [control]


## STEP 6 — Evaluation: post-fit diagnostics

Diagnostics on the fitted mixed models. Residual non-normality and
heteroskedasticity are expected with bounded socioeconomic rates and are handled
in the FE model by clustered SEs; they are reported here for completeness. The
random-intercept normality check is a soft assumption — mild departures do not
invalidate the fixed-effect inference, which is the reported result.

In [18]:
# ═════════════════════════════════════════════════════════════
# STEP 6 — Evaluation: post-fit diagnostics
# ═════════════════════════════════════════════════════════════
def evaluate(res):
    me, Y = res["me"], res["Y"]
    resid  = np.asarray(me.resid)
    fitted = np.asarray(me.fittedvalues)

    jb_p = float(stats.jarque_bera(resid)[1])
    re_vals = np.array([float(v.iloc[0]) for v in me.random_effects.values()])
    re_p = (float(stats.shapiro(re_vals)[1]) if len(re_vals) < 5000
            else float(stats.normaltest(re_vals)[1]))
    bp_p = float(sm.OLS(resid**2, sm.add_constant(fitted)).fit().f_pvalue)

    print(f"[{OUTCOMES[Y]}]  eval")
    print(f"   converged = {me.converged}   AIC = {me.aic:.1f}   BIC = {me.bic:.1f}")
    print(f"   residual normality   JB p = {jb_p:.3g}  "
          f"{'(non-normal — handled by clustered SE in FE)' if jb_p<0.05 else '(ok)'}")
    print(f"   random-intercept norm p = {re_p:.3g}  "
          f"{'(mild departure)' if re_p<0.05 else '(ok)'}")
    print(f"   homoskedasticity     BP p = {bp_p:.3g}  "
          f"{'(heterosked. — handled by clustered SE in FE)' if bp_p<0.05 else '(ok)'}")
    print(f"   ICC = {res['icc']:.3f}  -> {res['icc']*100:.0f}% of variance is "
          f"between-regency")
    print()

evaluate(res_pov)
evaluate(res_tpt)

[Poverty rate (%)]  eval
   converged = True   AIC = -11763.4   BIC = -11652.1
   residual normality   JB p = 4.72e-308  (non-normal — handled by clustered SE in FE)
   random-intercept norm p = 0.0065  (mild departure)
   homoskedasticity     BP p = 5.7e-34  (heterosked. — handled by clustered SE in FE)
   ICC = 0.988  -> 99% of variance is between-regency

[Unemployment rate / TPT (%)]  eval
   converged = True   AIC = -622.8   BIC = -511.5
   residual normality   JB p = 0  (non-normal — handled by clustered SE in FE)
   random-intercept norm p = 0.000183  (mild departure)
   homoskedasticity     BP p = 1.11e-64  (heterosked. — handled by clustered SE in FE)
   ICC = 0.697  -> 70% of variance is between-regency



## STEP 7 — What to report

**Poverty.** The fixed-effects model finds no significant association between any
flood dimension and the poverty rate, and the flood-block joint test is
non-significant. The mixed-effects specification yields the same near-zero,
non-significant flood coefficients and a non-significant flood block, so the null
is not an artefact of treating regency heterogeneity as fixed. The very high ICC
reflects that poverty differences are overwhelmingly cross-sectional — between
regencies — which is expected.

**Unemployment (TPT).** The fixed-effects results are suggestive but fragile — no
flood dimension is robustly significant, and the flood-block joint test is
non-significant. The mixed-effects estimates for the flood variables are close to
the FE estimates and remain non-significant (event count is marginal in FE and
not significant in mixed), and the mixed flood-block is likewise non-significant.
This is consistent with reporting TPT as fragile rather than as an established
effect.

**For both outcomes:** the flood conclusion **agrees** across the FE and mixed
specifications — that agreement is the point of the check. Control coefficients
may move because they are dominated by between-regency variation (section 5); this
does not affect the flood conclusions. Fixed effects remains the main inferential
specification (per the Mundlak test in the FE notebooks); the mixed model is a
robustness check.

> The mixed-effects check does not turn either null/fragile result into an effect.
> Report the numbers printed above; do not describe either outcome as showing a
> flood effect.